<a href="https://colab.research.google.com/github/edilsoncarrasco/IA-laboratorio7/blob/main/unidad-3-transformers/Lab03_Arquitectura_BERT_BETO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP + Transformers


---
## 1) Visualización de la Configuración de BERT 👀

In [1]:
#!pip install transformers

In [2]:
from transformers import BertConfig, BertModel

Cargar la configuración del modelo BERT en español

In [3]:
config = BertConfig.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
print("Número de capas de BERT:", config.num_hidden_layers)

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

Número de capas de BERT: 12


Cargar el modelo para visualizar su arquitectura

In [4]:
model = BertModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", output_hidden_states=True)
print(model)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31002, 768, padding_idx=1)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

## Análisis Sección 1

**1. Módulos principales**
- **embeddings (BertEmbeddings):** convierte los ids de tokens en vectores de 768 dimensiones sumando tres
  embeddings: `word_embeddings` (vocabulario ≈ 31002 × 768), `position_embeddings` (512 × 768, posición del
  token) y `token_type_embeddings` (2 × 768, segmento A/B). Luego aplica `LayerNorm` y `Dropout`.
- **encoder (BertEncoder):** `ModuleList` de **12 bloques `BertLayer`** apilados. Cada uno contiene
  autoatención multi-cabeza (12 cabezas) y una red Feed-Forward. Procesa toda la secuencia en paralelo
  y de forma bidireccional.
- **pooler (BertPooler):** toma el vector del token `[CLS]` y le aplica `Linear(768, 768)` + `Tanh`,
  generando una representación de toda la oración (útil para clasificación).

**2. Subcapas dentro de una BertLayer**
- **Atención (`attention.self`):** `query`, `key` y `value`, cada una `Linear(768 → 768)`. Generan Q, K y V.
  Se dividen en 12 cabezas de 64 dimensiones (768/12).
- **Proyección de salida (`attention.output.dense`):** `Linear(768 → 768)` que recombina las cabezas.
- **Residual + normalización:** `attention.output.LayerNorm` (Add & Norm tras la atención) y
  `output.LayerNorm` (Add & Norm tras el Feed-Forward). Las conexiones residuales suman la entrada
  de la subcapa a su salida antes de normalizar.
- **Feed-Forward:** `intermediate.dense` (768 → 3072, activación GELU) y `output.dense` (3072 → 768).

**3. Dimensión oculta:** `hidden_size = d_model = 768` (con intermedia de 3072 = 4 × 768).


## 2) Visualización de las Capas y Hidden States 🔍

In [5]:
from transformers import BertTokenizer
import torch

Cargar tokenizer y modelo BERT en español

In [6]:
tokenizer = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model = BertModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", output_hidden_states=True)

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Texto de ejemplo

In [7]:
text = "El producto de Samsung Galaxy S21 llegó el 12 de marzo y superó mis expectativas."
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
outputs = model(**inputs)

In [8]:
print(outputs)

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.3688, -0.2218, -0.6865,  ..., -0.4430,  0.2883,  0.1794],
         [-0.2616, -0.0662, -0.6243,  ..., -0.5904,  0.6507,  0.2768],
         [-0.2415, -0.2261, -0.6785,  ..., -0.1096,  0.0910,  0.5395],
         ...,
         [-1.4393, -0.1986, -0.5770,  ..., -0.5261,  0.0716,  1.0376],
         [-0.7026,  0.3509, -0.4953,  ..., -0.3322,  0.3734,  0.1284],
         [-1.2273, -0.4597, -1.0296,  ..., -0.6864,  0.9237, -0.5841]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[-2.9178e-02, -1.2531e-01, -6.7262e-02, -3.4863e-01,  6.4434e-02,
          1.8262e-01, -3.2281e-01, -3.8069e-01, -6.1404e-02, -5.0928e-01,
         -1.2397e-01,  2.4272e-01,  4.3507e-01, -2.7007e-01,  8.9825e-02,
          6.0054e-01,  2.7781e-01,  6.2875e-01,  5.3873e-01, -2.2302e-01,
          2.0932e-01, -5.3921e-01,  1.5343e-01, -1.7560e-01,  1.4352e-01,
         -5.5585e-01, -3.8777e-02, -4.5974e-01, -5.9449e-01,  2.643

Extraer los hidden states


In [9]:
hidden_states = outputs.hidden_states
print("Número de capas (incluyendo la capa de embedding):", len(hidden_states))

Número de capas (incluyendo la capa de embedding): 13


In [10]:
print(inputs)                          # diccionario completo
print(inputs["input_ids"])
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print(inputs["token_type_ids"])
print(inputs["attention_mask"])
print("Número de hidden_states:", len(hidden_states))
print("Shape hidden_states[0]:", hidden_states[0].shape)
print("Shape hidden_states[-1]:", hidden_states[-1].shape)

{'input_ids': tensor([[    4,  1162,  4196,  1008,  4262,  8643, 16685, 21291, 24010,  1057,
          8926,  4444,  1040,  1797,  1008,  3008,  1042, 27994,  1354, 12956,
          1009,     5]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
tensor([[    4,  1162,  4196,  1008,  4262,  8643, 16685, 21291, 24010,  1057,
          8926,  4444,  1040,  1797,  1008,  3008,  1042, 27994,  1354, 12956,
          1009,     5]])
['[CLS]', 'El', 'producto', 'de', 'Sam', '##su', '##ng', 'Gala', '##xy', 'S', '##21', 'llegó', 'el', '12', 'de', 'marzo', 'y', 'superó', 'mis', 'expectativas', '.', '[SEP]']
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
Número de hidden_states: 13
Shape hidden_states[0]: torch.Size([1, 22, 768])
Shape hidden_states[-1

## Análisis Sección 2

**1. Diccionario `inputs`**
- **`input_ids`:** ids enteros de cada token según el vocabulario de BETO. Incluye `[CLS]` al inicio (id 4) y
  `[SEP]` al final (id 5). Las palabras poco frecuentes se dividen en subpalabras (WordPiece, con `##`):
  "Samsung" → `Sam`, `##su`, `##ng`; "Galaxy" → `Gala`, `##xy`; "S21" → `S`, `##21`.
  Por eso 15 palabras producen 22 tokens.
- **`token_type_ids`:** indica a qué segmento pertenece cada token (0 = oración A, 1 = oración B). Sirve
  para tareas con pares de oraciones. Aquí hay una sola oración, así que todo es 0.
- **`attention_mask`:** 1 = token real que debe atenderse, 0 = relleno (padding) que se ignora. Todos son 1
  porque se procesa una sola oración sin padding.

**2. ¿Por qué 13 hidden_states si hay 12 capas?**
El tensor en la posición 0 es la **salida de la capa de embeddings** (embeddings de palabra + posición +
segmento, con LayerNorm), es decir, la entrada al primer encoder. Las posiciones 1 a 12 son las salidas de
cada uno de los 12 `BertLayer`. Total: 1 + 12 = 13.

**3. `hidden_states[0].shape` = `torch.Size([1, 22, 768])`**
- **batch_size = 1:** una sola oración.
- **sequence_length = 22:** número de tokens tras tokenizar: 20 de la oración más `[CLS]` y `[SEP]`.
- **hidden_size = 768:** dimensión del vector que representa cada token.

`hidden_states[-1].shape` también es `torch.Size([1, 22, 768])`: las capas del encoder conservan la forma
del tensor y solo cambian los valores (cada vector se contextualiza con los demás tokens).

### 3) Diagrama Conceptual y Pseudocódigo 🙌

### Diagrama conceptual del flujo de Self-Attention

```text
X (hidden_states[0], [22 × 768])
 ├── W_Q ──► Q ──┐
 ├── W_K ──► K ──┴─► Q·Kᵀ / √d_k ─► (+ máscara) ─► softmax ─► pesos [22 × 22] ─┐
 └── W_V ──► V ────────────────────────────────────────────────────────────────┴─► pesos · V
                                                                                    │
                                                              W_O (attention.output.dense)
                                                                                    │
                                                                     Add & Norm (X + salida)
                                                                                    │
                                                              Feed-Forward ─► Add & Norm ─► H
```

## Pseudocódigo de Self-Attention

Fórmula:

$$Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

```
ENTRADA: X de forma [seq_len, d_model]   (embeddings de los tokens)
PARÁMETROS: W_Q, W_K, W_V de forma [d_model, d_k]

1. Q = X · W_Q          # proyección lineal: qué busca cada token
2. K = X · W_K          # proyección lineal: qué ofrece cada token
3. V = X · W_V          # proyección lineal: contenido a combinar
4. scores = Q · Kᵀ      # [seq_len, seq_len]: similitud de cada token con todos
5. scores = scores / √d_k   # escala para evitar valores grandes y gradientes
                            # que saturen el softmax
6. scores = scores + máscara   # (BERT) tokens de padding reciben -∞
7. pesos = softmax(scores)     # por fila: cada fila suma 1
8. salida = pesos · V          # suma ponderada de V: [seq_len, d_k]

MULTI-CABEZA (BERT-base): repetir 1-8 con h = 12 cabezas, d_k = 768/12 = 64,
concatenar las 12 salidas → [seq_len, 768] y aplicar W_O (attention.output.dense).
```


**Demostración en PyTorch:** usa matrices `W_Q, W_K, W_V` **aleatorias** (`torch.randn`), por lo que solo
valida las dimensiones y las propiedades del cálculo (`pesos` de `[22, 22]`, salida de `[22, 64]`,
filas que suman 1), no los valores reales de BETO. BETO usa 12 cabezas de 64 dimensiones con pesos
aprendidos (`attention.self.query/key/value`).

In [11]:
import torch, math
import torch.nn.functional as F

X = hidden_states[0][0]                 # [N, 768]
d_k = 64
W_Q, W_K, W_V = (torch.randn(768, d_k) for _ in range(3))
Q, K, V = X @ W_Q, X @ W_K, X @ W_V
scores = (Q @ K.T) / math.sqrt(d_k)
pesos = F.softmax(scores, dim=-1)
salida = pesos @ V
print(pesos.shape, salida.shape)        # [N, N] y [N, 64]

torch.Size([22, 22]) torch.Size([22, 64])


## Relación con BertSelfAttention

El diagrama y el pseudocódigo van al cierre porque primero se necesita observar la arquitectura real
(Sección 1) y los tensores que fluyen por ella (Sección 2); solo entonces se puede explicar la lógica
interna que los produce. Es un cierre de lo concreto a lo abstracto.

Se relacionan directamente con `BertSelfAttention` (`attention.self`) que se vio en `print(model)`:
- Las capas `query`, `key` y `value` (`Linear(768, 768)`) son las proyecciones W_Q, W_K, W_V de los
  pasos 1-3 (las 12 cabezas de 64 dimensiones se calculan juntas en una sola matriz de 768).
- El producto Q·Kᵀ, el escalado por √64, el softmax y la multiplicación por V (pasos 4-8) ocurren dentro
  del forward de esa clase, con `dropout` sobre los pesos de atención.
- `attention.output.dense` + `LayerNorm` implementan W_O y el Add & Norm posterior.
